# Guanajuato WOFOST 8.1 crop and water scenarios

This notebook runs one crop and variety for each season using an immutable `farm_profiles` run. `WATER_MODE` selects potential production (`potential`), rainfall with a water balance (`rainfed`), or a water balance with dated irrigation (`irrigated`). The reference point (20.4243195, -101.1466639) is **not a surveyed field**. Sowing dates, variety, elevation override, CO₂, soil, and irrigation inputs are manual scenarios unless separately measured.

The cell below prints the supported WOFOST crops and the catalog varieties for the selected crop. These are model parameter sets, **not crop suitability recommendations**. For `rainfed` or `irrigated`, supply all soil parameters and initial available water. For `irrigated`, also supply a system name and dated events with depth and application efficiency. The system name records provenance; depth and efficiency determine the model input. Do not fill real-data runs with invented soil values.

Install dependencies with `pip install -r requirements.txt`. Configure Application Default Credentials for GCS. PCSE crop parameters may need network access on first use. See the [WOFOST 8.1 catalog](https://raw.githubusercontent.com/ajwdewit/WOFOST_crop_parameters/wofost81/crops.yaml).

Set `USE_SYNTHETIC_SMOKE=True` to check the wiring with invented weather. Synthetic outputs are not Guanajuato observations or calibration evidence.

## SIAP error check

The last cells compare uncalibrated WOFOST yield with SIAP municipal production per sown hectare for Valle de Santiago. Add the exact SIAP crop label to `SIAP_CROP_LABELS` when changing crops. Spatial scale and grain moisture basis may differ. MAE, RMSE, mean bias, and MAPE are diagnostic comparisons, not field validation.

In [ ]:
import sys
from pathlib import Path
repo_root = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
                  if (candidate / "gradientdescentfarmers" / "model_builder").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook from within the repository or install the package")
sys.path.insert(0, str(repo_root))

In [ ]:
from pcse.input import YAMLCropDataProvider
from pcse.models import Wofost81_PP, Wofost81_WLP_CWB
from gradientdescentfarmers.model_builder.gcs import GCSDataClient
from gradientdescentfarmers.model_builder.wofost import SUPPORTED_CROPS, available_crop_varieties, run_wofost

USE_SYNTHETIC_SMOKE = False  # True runs invented weather
RUN_URI = "gs://data_dump_samples/farm_profiles/guanajuato_valle_demo/runs/67b66942681a4214"
CROP = "maize"
VARIETY = "Grain_maize_202"
WATER_MODE = "potential"  # potential, rainfed, or irrigated
SEASONS = [("2024-05-15", "2024-12-15"), ("2025-05-15", "2025-12-15")]
CO2_PPM = 420.0  # scenario assumption
ELEVATION_OVERRIDE_M = None  # use profile topography when available
SIAP_CROP_LABELS = {"maize": "Maíz grano"}  # add an exact SIAP label for each selected crop

# Required for rainfed and irrigated runs. Use measured or independently justified values.
# Fractions are m3/m3; RDMSOL is cm; SOPE, KSUB, K0 are cm/day.
SOIL_PARAMETERS = {"SMW": None, "SMFCF": None, "SM0": None, "RDMSOL": None,
                   "CRAIRC": None, "SOPE": None, "KSUB": None, "K0": None}
INITIAL_AVAILABLE_WATER_CM = None  # WAV at the start of each season
IRRIGATION_SYSTEM = None  # e.g. drip or sprinkler; provenance only
IRRIGATION_EVENTS_BY_YEAR = {
    2024: [],  # e.g. {"date": "2024-06-01", "amount_mm": 40, "efficiency": 0.8}
    2025: [],
}
FIELD_OBSERVATIONS = {"observed_field_id": None, "observed_harvest_t_ha": None,
                      "commercial_grain_moisture_pct": None}

catalog_model = Wofost81_PP if WATER_MODE == "potential" else Wofost81_WLP_CWB
catalog = available_crop_varieties(YAMLCropDataProvider(model=catalog_model))
print("Available crops:", sorted(catalog))
assert CROP in catalog, f"Choose CROP from {sorted(catalog)}"
print(f"Available varieties for {CROP}:", catalog[CROP])
assert VARIETY in catalog[CROP], f"Choose VARIETY from {catalog[CROP]}"
assert CROP in SIAP_CROP_LABELS, "Add the selected crop's exact SIAP label to SIAP_CROP_LABELS"
assert WATER_MODE in ("potential", "rainfed", "irrigated")

In [ ]:
import pandas as pd
if USE_SYNTHETIC_SMOKE:
    # A wiring check only: these values are invented, not Guanajuato observations.
    dates = pd.date_range("2024-01-01", "2025-12-31")
    weather = pd.DataFrame({"date": dates, "tmin_c": 12.0, "tmax_c": 27.0,
                            "precip_mm": 2.0, "rh_pct": 65.0, "wind_m_s": 2.0,
                            "solar_mj_m2_day": 19.0})
    site = {"latitude": 20.4243195, "longitude": -101.1466639}
    elevation = 1700.0  # invented for smoke test
    run_id = "SYNTHETIC_SMOKE"
else:
    assert "REPLACE_WITH_RUN_ID" not in RUN_URI, "Set an immutable run URI before using GCS"
    gcs = GCSDataClient(project="gdfarming")
    profile = gcs.load_profile(RUN_URI)
    weather = profile.table("weather_daily")
    site = profile.manifest["farmer_inputs"]
    assert abs(site["latitude"] - 20.4243195) < 0.01
    assert abs(site["longitude"] + 101.1466639) < 0.01
    elevation = ELEVATION_OVERRIDE_M if ELEVATION_OVERRIDE_M is not None else profile.manifest.get("topography", {}).get("elevation_m")
    assert elevation is not None, "Supply a measured/explicit elevation if profile topography is missing"
    run_id = profile.manifest["run_id"]
print(run_id, weather.shape, elevation)

In [ ]:
import pandas as pd
simulation_df = pd.concat([
    run_wofost(
        weather, crop=CROP, variety=VARIETY, sowing_date=start, end_date=end,
        latitude=site["latitude"], longitude=site["longitude"],
        elevation_m=float(elevation), co2_ppm=CO2_PPM, run_id=run_id,
        water_mode=WATER_MODE,
        soil_parameters=None if WATER_MODE == "potential" else SOIL_PARAMETERS,
        initial_available_water_cm=None if WATER_MODE == "potential" else INITIAL_AVAILABLE_WATER_CM,
        irrigation_system=IRRIGATION_SYSTEM if WATER_MODE == "irrigated" else None,
        irrigation_events=IRRIGATION_EVENTS_BY_YEAR.get(pd.Timestamp(start).year, [])
            if WATER_MODE == "irrigated" else None,
    )
    for start, end in SEASONS
], ignore_index=True)
display(simulation_df.tail())
display(simulation_df.loc[simulation_df.is_final_day,
                          ["year", "crop", "variety", "water_mode", "irrigation_system",
                           "irrigation_applied_mm", "yield_t_ha"]])
from pathlib import Path
scenario_tag = f"{WATER_MODE}_{CROP}"
output_name = f"guanajuato_synthetic_{scenario_tag}.csv" if USE_SYNTHETIC_SMOKE else f"guanajuato_{scenario_tag}.csv"
OUTPUT_CSV = repo_root / "data" / "model_builder" / output_name
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
simulation_df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

## Compare the selected scenario with SIAP

Positive signed error means WOFOST predicts more than SIAP. SIAP rows are municipal aggregates. Both seasons need matching crop and year observations. Changing `WATER_MODE` produces a separate comparison file so calibration uses the intended scenario.

In [ ]:
from gradientdescentfarmers.model_builder.calibration import compare_siap_yield, error_metrics
MUNICIPALITY = "Valle de Santiago"  # Carmelitas Chico locality
SIAP_PREFIX = "gs://data_dump_samples/SIAP_Agricultural_Production_2000_2025/"
SIAP_URIS = [
    f"{SIAP_PREFIX}Municipal (2003_2025)/Cierre_agricola_mun_{year}.csv"
    for year in sorted(simulation_df.loc[simulation_df.is_final_day, "year"].unique())
]
if USE_SYNTHETIC_SMOKE:
    finals = simulation_df.loc[simulation_df.is_final_day, ["year", "yield_t_ha"]]
    siap_df = pd.DataFrame({"year": finals.year, "municipality": "TEST",
                            "crop": SIAP_CROP_LABELS[CROP], "unit": "Tonelada",
                            "sown_ha": 10.0, "production_volume": finals.yield_t_ha.astype(float) * 5})
    municipality_for_check = "TEST"
else:
    siap_df = pd.concat([gcs.read_frame(uri) for uri in SIAP_URIS], ignore_index=True)
    municipality_for_check = MUNICIPALITY
baseline_comparison_df = compare_siap_yield(
    simulation_df, siap_df, municipality=municipality_for_check, crop_labels=SIAP_CROP_LABELS
)
baseline_metrics_df = error_metrics(baseline_comparison_df, prediction_column="simulated_t_ha")
baseline_metrics_df.insert(0, "crop", CROP)
baseline_metrics_df.insert(1, "water_mode", WATER_MODE)
display(baseline_comparison_df[["year", "crop", "water_mode", "observed_t_ha",
                                "simulated_t_ha", "raw_error_t_ha", "raw_absolute_error_t_ha",
                                "raw_absolute_percentage_error"]])
display(baseline_metrics_df)
comparison_name = f"siap_synthetic_{scenario_tag}_comparison.csv" if USE_SYNTHETIC_SMOKE else f"siap_{scenario_tag}_comparison.csv"
COMPARISON_CSV = repo_root / "data" / "model_builder" / comparison_name
baseline_comparison_df.to_csv(COMPARISON_CSV, index=False)
print("Saved:", COMPARISON_CSV)